# Day 2 · 2교시 · 환경 · 모델 반입

> **VLM 경량화 2일 과정 · Day 2 (2교시) · 실습**
> 실습 환경: **RunPod A100 80GB** · 베이스 이미지: `runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04`

---

## 이 교시의 목표
- RunPod A100에 **uv + Python 3.12 + venv**(양자화용)를 만들고 **`llmcompressor`** 를 설치한다.
- Day1에서 Hub에 올린 **병합 4B를 반입**하고 베이스 **8B를 다운로드**한다.
- ChartQA에서 **calibration set**(256개)을 준비한다.

| 규약 | 값 |
|---|---|
| 작업 폴더 | `/workspace/VLM_FT_2` |
| 양자화 venv | `/workspace/VLM_FT_2/.venv-quant` (Python 3.12) |
| torch | 2.8.0 + torchvision 0.23.0 (cu128, 베이스 이미지에 맞춤) |
| 서빙 venv | **Day2-5에서 별도 생성**(vLLM ↔ llmcompressor 의존성 충돌 방지) |


## 0. 공통 헤더 — RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN 로드

> 📌 **모든 Day 2 노트북은 이 셀을 먼저 실행합니다.** RunPod 영구 볼륨의 작업 폴더 **`/workspace/VLM_FT_2`** 를 기준으로 잡고, `.env`의 **HF_TOKEN**을 불러옵니다. (Day2는 RunPod이라 Google Drive 마운트가 없습니다.)
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(모델·AWQ 결과·평가/벤치 JSON이 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다. `login()`은 호출하지 않습니다(환경변수만으로 충분).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN(.env) 로드
#  (모든 Day2 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) RunPod 영구 볼륨의 작업 폴더 VLM_FT_2 (교시 간 모델·결과 공유). Drive 마운트 불필요.
VLM_DIR = '/workspace/VLM_FT_2'
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 모델만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /workspace/VLM_FT_2


## 1. 양자화용 venv 생성 + 커널 등록

아래 셀은 **현재(베이스) 커널**에서 실행합니다. `uv`로 Python 3.12 venv를 만들고, torch(cu128)·transformers·llmcompressor를 설치한 뒤 **Jupyter 커널로 등록**합니다.

> ⏱️ 설치에 수 분 걸립니다(torch가 큼). 끝나면 **커널을 전환**합니다(다음 안내).

In [2]:
# ── (베이스 커널에서 실행) uv venv + 설치 + 커널 등록 ────────
import subprocess

WORKDIR = VLM_DIR   # 상단 공통 헤더의 작업 폴더 변수 사용
VENV = f'{WORKDIR}/.venv-quant'
PY = f'{VENV}/bin/python'

def run(cmd: str):
    """셸 명령 실행 + 출력 표시."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout[-1500:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1500:])
    return r.returncode

run(f'mkdir -p {WORKDIR}')
run('pip install -q uv')                                  # uv 설치(베이스에)
run(f'uv venv --python 3.12 {VENV}')                      # Python 3.12 venv

#   - --torch-backend=auto : pod 드라이버 CUDA에 맞는 torch 자동 선택(12.8 pod→cu12x, 13 pod→cu13x) → cu130 미스매치 방지
#   - peft·qwen-vl-utils   : Day2-3의 'LoRA 병합 + Qwen3-VL' 용
#   - torchvision          : qwen-vl-utils가 내부적으로 import(없으면 ModuleNotFoundError) — torch와 짝 맞춰 자동 설치
run(f'uv pip install --python {PY} --torch-backend=auto '
    f'llmcompressor peft qwen-vl-utils torchvision accelerate datasets '
    f'python-dotenv huggingface_hub ipykernel ipywidgets "pillow<12"')

# 이 venv를 Jupyter 커널로 등록
run(f'{PY} -m ipykernel install --user --name vlm_quant --display-name "Python (vlm_quant)"')
print('\n완료 — 이제 커널을 전환하세요(다음 셀 안내).')

$ mkdir -p /workspace/VLM_FT_2
$ pip install -q uv
$ uv venv --python 3.12 /workspace/VLM_FT_2/.venv-quant
$ uv pip install --python /workspace/VLM_FT_2/.venv-quant/bin/python --torch-backend=auto llmcompressor peft qwen-vl-utils torchvision accelerate datasets python-dotenv huggingface_hub ipykernel ipywidgets "pillow<12"
$ /workspace/VLM_FT_2/.venv-quant/bin/python -m ipykernel install --user --name vlm_quant --display-name "Python (vlm_quant)"
Installed kernelspec vlm_quant in /root/.local/share/jupyter/kernels/vlm_quant


완료 — 이제 커널을 전환하세요(다음 셀 안내).


## 2. ⚠️ 커널 전환

JupyterLab 우상단 커널 선택 → **`Python (vlm_quant)`** 으로 바꾼 뒤, **이 아래 셀부터** 실행하세요. (이후 Day2 노트북들도 이 커널로 엽니다 — 그 노트북들은 맨 위 공통 헤더가 vlm_quant에서 바로 실행되므로 추가 작업이 없습니다.)

> 커널을 바꾸면 위 공통 헤더에서 만든 변수(`VLM_DIR`·`HF_TOKEN` 등)가 **새 커널엔 없습니다.** 그래서 **바로 아래 검증 셀이 헤더 설정을 vlm_quant 커널에서 다시 한 번 적용**합니다.

In [1]:
# ── (vlm_quant 커널에서 실행) 헤더 재설정 + 환경 검증 ───────
# ⚠️ 커널을 전환했으므로 상단 공통 헤더의 변수가 새 커널엔 없습니다 → 동일하게 다시 설정.
import os
VLM_DIR = '/workspace/VLM_FT_2'
DATA_DIR = f'{VLM_DIR}/data'
ENV_PATH = f'{VLM_DIR}/.env'
os.makedirs(DATA_DIR, exist_ok=True)
try:
    from dotenv import load_dotenv
    if os.path.exists(ENV_PATH): load_dotenv(ENV_PATH)
except ImportError:
    pass
print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음', '| VLM_DIR =', VLM_DIR)

import sys, torch, transformers
print('python   :', sys.version.split()[0])           # 3.12.x 여야 정상
print('torch    :', torch.__version__)                # 드라이버에 맞는 cu12x/cu13x
print('cuda     :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('transformers:', transformers.__version__)
import llmcompressor
print('llmcompressor:', llmcompressor.__version__)

HF_TOKEN: 로드됨 | VLM_DIR = /workspace/VLM_FT_2
python   : 3.12.13
torch    : 2.12.0+cu130
cuda     : True NVIDIA A100 80GB PCIe
transformers: 5.10.1
llmcompressor: 0.12.0


## 3. 모델 반입 — Day1 병합 4B + 베이스 8B

Day1(Colab)에서 Hub에 올린 **병합 4B**와, 비교군인 **베이스 8B**를 받아옵니다. `snapshot_download`는 디스크에만 받고 GPU엔 올리지 않아 빠릅니다.

1. HF **read 토큰** 인증 — 아래 셀이 `/workspace/VLM_FT_2/.env` 에서 토큰을 읽습니다(없으면 최초 1회 입력 후 저장 → 이후 자동 로드). 병합 4B가 비공개라 필요합니다.
2. `MERGED_REPO`를 본인 저장소로 수정

In [ ]:
# ── HF 로그인 + 스냅샷 다운로드 ──────────────────────────────
# HF_TOKEN 은 상단 공통 헤더가 .env에서 로드. 없으면(최초 1회) 입력받아 ENV_PATH에 저장.
import os
if not os.environ.get('HF_TOKEN'):
    from getpass import getpass
    from dotenv import load_dotenv
    tok = getpass('HF 토큰 입력(최초 1회): ').strip()
    with open(ENV_PATH, 'w') as f:
        f.write(f'HF_TOKEN={tok}\n')
    load_dotenv(ENV_PATH)
    print('.env 생성:', ENV_PATH)
assert os.environ.get('HF_TOKEN'), '.env 에 HF_TOKEN 이 없습니다'
print('HF 토큰 준비 완료 (.env 환경변수)')
from huggingface_hub import snapshot_download

BASE_4B      = 'Qwen/Qwen3-VL-4B-Instruct'
ADAPTER_REPO = 'leeunzin/Qwen3-VL-4B-ChartQA-lora'   # ← Day1-5에서 올린 어댑터(비공개) /root/.cache/huggingface에 다운로드 
BASE_8B      = 'Qwen/Qwen3-VL-8B-Instruct'

# 4B는 'base + 어댑터'를 받아 Day2-3에서 병합(A100), 8B는 베이스 그대로
path_4b_base = snapshot_download(BASE_4B)       # 베이스 4B
path_adapter = snapshot_download(ADAPTER_REPO)  # Day1 LoRA 어댑터(비공개)
path_8b      = snapshot_download(BASE_8B)       # 베이스 8B
print('4B base 경로 :', path_4b_base)
print('어댑터 경로  :', path_adapter)
print('8B base 경로 :', path_8b)
print('→ 4B 병합은 Day2-3(AWQ 직전)에서 수행합니다')

HF 토큰 준비 완료 (.env 환경변수)


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

4B base 경로 : /root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17
어댑터 경로  : /root/.cache/huggingface/hub/models--nugunaai--Qwen3-VL-4B-ChartQA-lora/snapshots/1952db1ab7d5335c45352daa115e59e252ab541d
8B base 경로 : /root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b
→ 4B 병합은 Day2-3(AWQ 직전)에서 수행합니다


In [3]:
# ── 로드 확인(8B) — 인스턴스화·파라미터 수 점검 후 해제 ──────
import gc
from transformers import Qwen3VLForConditionalGeneration

# A100은 bf16 지원 → 양자화 입력은 bf16/fp16로 로드(여기선 확인만)
# path_8b는 상단에서 다운로드한 베이스 8B 모델 경로
m = Qwen3VLForConditionalGeneration.from_pretrained(
    path_8b,                          # 베이스 8B 모델 경로
    dtype=torch.bfloat16,             # A100 최적화: bf16으로 로드
    device_map='cuda')                # GPU에 로드
    
# 모델 파라미터 수 계산(단위: 십억)
n_params = sum(p.numel() for p in m.parameters()) / 1e9
print(f'8B 로드 OK | 파라미터: {n_params:.2f} B | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

# 확인 끝났으니 해제(Day2-3에서 다시 로드하므로 메모리 절약)
del m                                 # 모델 객체 삭제
gc.collect()                          # 가비지 컬렉션 실행
torch.cuda.empty_cache()              # GPU 캐시 정리
print('해제 완료')

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

8B 로드 OK | 파라미터: 8.77 B | VRAM: 16.3 GB
해제 완료


## 4. Calibration set 준비

1교시에서 본 대로, AWQ는 **대표 입력 몇 백 개**로 채널별 활성값 통계를 모읍니다. 배포 분포와 맞춰 **ChartQA train에서 256개**를 시드 고정으로 뽑습니다.

| 항목 | 값 | 이유 |
|---|---|---|
| 개수 | **256** | 128~512 권장 범위의 중간(품질·시간 균형) |
| 출처 | ChartQA train | 배포 대상과 동일 분포 |
| 형식 | 이미지+질문 채팅 메시지 | 멀티모달 calibration |

In [ ]:
# ── ChartQA calibration 서브셋 준비 ───────────────────────────────
from datasets import load_dataset

# AWQ calibration에 사용할 샘플 개수
N_CALIB = 256

# ChartQA train split 로드 → 재현 가능하도록 seed 고정 셔플 → 앞에서 N_CALIB개 선택
calib_raw = (
    load_dataset('HuggingFaceM4/ChartQA', split='train')
    .shuffle(seed=42)
    .select(range(N_CALIB))
)
print('calibration 샘플 수:', len(calib_raw))

# 첫 번째 샘플 확인 (실제 토크나이즈/전처리는 Day2-3에서 수행)
ex = calib_raw[0]

# 멀티모달 입력 메시지 포맷 미리보기
# - image: 실제 실행 시 PIL 이미지 객체가 들어감
# - text : ChartQA의 query(질문)
preview_messages = [{
    'role': 'user',
    'content': [
        {'type': 'image', 'image': '<PIL 이미지>'},
        {'type': 'text',  'text': ex['query']},
    ],
}]

# JSON 형태로 보기 좋게 출력 (한글 깨짐 방지: ensure_ascii=False)
import json as _json
print(_json.dumps(preview_messages, indent=2, ensure_ascii=False))

# 샘플 이미지 해상도 확인
print('이미지 크기:', ex['image'].size)

calibration 샘플 수: 256
[
  {
    "role": "user",
    "content": [
      {
        "type": "image",
        "image": "<PIL 이미지>"
      },
      {
        "type": "text",
        "text": "What is the sales performance of accessories by Level/flat?"
      }
    ]
  }
]
이미지 크기: (800, 557)


## 5. 정리 + 다음 교시 예고

- **양자화 venv**(uv·Python 3.12·llmcompressor)를 만들고 커널로 등록했습니다.
- **병합 4B·베이스 8B**를 디스크에 받아두었습니다(`path_4b`, `path_8b`).
- **calibration set 256개**(`calib_raw`)를 준비했습니다.

### 다음 교시 — Day2-3 · AWQ 생성 (핵심)
`AWQModifier` + `QuantizationModifier`(W4A16, group128, 비전 `ignore`)로 **4B·8B를 각각 AWQ 4bit로 변환**하고(`oneshot`), 모델 크기를 원본과 비교합니다.

> ✅ **체크포인트**: ① vlm_quant 커널에서 torch.cuda가 True ② `path_4b`/`path_8b`가 잡혔다 ③ `calib_raw` 256개를 준비했다.